<div style="display: flex; align-items: center; justify-content: center; gap: 12px;">
    <h1 style="margin: 0;"><b>Pokémon Game Simulation</b></h1>
    <img src="https://cdn-icons-png.flaticon.com/128/1752/1752776.png" width="50" height="50" alt="pokeball">
</div>

In this notebook, we will build a Pokémon Battle Simulator. We'll use the Complete Pokémon dataset from Kaggle, prepare it for processing, and train an ML algorithm to predict the match winner. Finally, we will build a turn-based battle simulator to simulate an actual game and see if our model can correctly guess the winner! 

### The roadmap

Every section below follows the typical flow of a machine learning project:

1. **Get & profile the data**: load the dataset and understand what each column means, which values are missing, and how the numbers are distributed.
2. **Clean it**: remove columns we don't need, fill in gaps, drop duplicates and tame extreme values.
3. **Transform & enrich it**: turn text categories into numbers, build new features that describe a Pokémon's battle strengths, and scale the numbers.
4. **Create battle data**: the dataset describes individual Pokémon, not fights, so we simulate 50,000 random battles to get examples where the winner is known.
5. **Train & evaluate a model**: teach a Random Forest to predict the winner from the two fighters' features, then measure how well it does on battles it has never seen.
6. **Play**: build a turn-based simulator and compare what happens in the "fight" with what the model predicted.

Happy coding!

# Import libraries

Before doing anything, we load the tools we need. Libraries are collections of ready-made code, so we don't have to write everything from scratch:

- **pandas** (`pd`): works with tables of data (called DataFrames). It's our main tool for loading, cleaning and reshaping the dataset.
- **numpy** (`np`): fast math and random number generation. We'll use it for the log transform and for the randomness inside battles.
- **seaborn** and **matplotlib** (`sns`, `plt`): plotting libraries that draw the charts which help us understand the data.

The machine learning tools from scikit-learn are imported later, right where we first use them.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Get the data

Here we load the dataset. The Complete Pokémon dataset from Kaggle is a CSV file (`pokemon.csv`) where **every row is one Pokémon** and every column describes it: its stats (HP, attack, speed...), its types, how much damage it takes from each attack type (the `against_*` columns), and more.

Instead of downloading the file by hand, we use the `kagglehub` library. Like `load_iris()` in scikit-learn, `kagglehub.dataset_load` fetches the data and returns it as a pandas DataFrame in one call. The file is cached after the first run, so it isn't downloaded again.

- `"rounakbanik/pokemon"` is the dataset's identifier on Kaggle (`owner/dataset-name`).
- `"pokemon.csv"` is the file we want from that dataset.
- `KaggleDatasetAdapter.PANDAS` tells it to return a pandas DataFrame. This needs the `kagglehub[pandas-datasets]` install.

We then keep the data under **two names**, `bk` and `df`:

- `df` is the working copy that we will clean and transform.
- `bk` ("backup") is a separate copy made with `df.copy()`, so changes to `df` never affect it. We'll need it later to look up the names and types that we remove from `df`.

In [ ]:
!pip install "kagglehub[pandas-datasets]" -q

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "rounakbanik/pokemon",
    "pokemon.csv",
)
bk = df.copy()

#### Preview the data

`head()` shows the first five rows. It's a quick sanity check that the file loaded correctly, and it lets us see the column names and what kind of values each column holds.

In [ ]:
df.head()

# Data Profiling

In this section, we will examine the data types of each column and understand its role and use case. We will also analyze column characteristics by type and inspect the correlations between numerical features.

**Why profile first?** A model can only be as good as the data we feed it. Profiling shows us what we're working with: how big the dataset is, which columns are numbers or text, where values are missing, and how the values are spread out. What we find here decides what we need to do in the cleaning and transformation sections.

#### Dataset size

`shape` returns `(rows, columns)`. Since each row is a Pokémon, the first number tells us how many Pokémon we have, and the second how many features describe each of them.

In [ ]:
df.shape

#### Column names

`columns` lists every feature in the dataset. Notice the `against_*` columns: each one is a damage multiplier against one attack type (for example, `against_fire = 2` means the Pokémon takes double damage from Fire moves). We'll rely on these when we simulate type advantages in battles.

In [ ]:
df.columns

#### Data types and non-null counts

`info()` shows, for every column, its data type and how many values are **not** missing. If a column has fewer non-null values than the number of rows, it has missing data. It also reveals columns stored with the wrong type, such as a numeric column that was read as text (we will meet this with `capture_rate`).

In [ ]:
df.info()

#### Summary statistics

`describe(include='all')` summarizes every column. For numeric columns it shows the mean, standard deviation, minimum, quartiles and maximum; for text columns it shows the number of unique values and the most frequent one. We use it to spot suspicious ranges and to get a feeling for the scale of each feature.

In [ ]:
df.describe(include='all')

### Check duplicates

Duplicated rows can bias a model: the same example would be counted twice, and copies could even end up in both the training and test sets. `duplicated()` marks every repeated row as `True`, and `sum()` counts them.

The count came out as zero, so there are no duplicates to handle in this dataset. (We run the same check again after cleaning, just to be safe.)

In [ ]:
df.duplicated().sum()

### Empty data

Most models can't work with missing values, so we need to know where they are. `df.isna()` turns the table into `True`/`False` (missing or not), and the heatmap draws it: **every colored mark is a missing value**. Vertical stripes show which columns have gaps. We'll decide how to fill each of them in the cleaning section.

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.isna())

### Correlation

The correlation heatmap shows how strongly each pair of numeric columns moves together, from -1 (opposite directions) through 0 (no relationship) to +1 (move together). `numeric_only=True` skips the text columns.

Why we care:
- Features that are highly correlated carry similar information (for instance, `base_total` is built from the individual stats, so we expect it to correlate with them).
- It's a quick way to see which features might be useful for predicting something, and which are redundant.

In [ ]:
plt.figure(figsize=(10,8))
corr = df.corr(numeric_only=True)
# make the heatmap ?

### Check the `type` column 

A count plot shows how many Pokémon belong to each primary type (`type1`). The categories are imbalanced, with some types much more common than others; however, this won't adversely affect our modeling. Class imbalance is mainly a problem when it happens in the **target** we are predicting. Here, type is only an input feature, and the target (who wins a battle) is created later from random pairings of Pokémon.

In [ ]:
sns.countplot(x=df['type1'])
plt.xticks(rotation=45)
plt.title('Count of Pokèmon by Type1')

### Check the stat data

The histograms show how each of the six base stats is distributed. Most of the stat data is positively skewed (right-skewed): many Pokémon have modest stats, and a few have very high ones, which creates a long tail on the right.

Why it matters: extreme values can dominate many models and make the numbers harder to compare. The goal is to compress the long right tail and pull extreme values closer to the center, making the distribution look more like a normal bell curve. We'll tackle this in a later section!

<div align="center">
    <img src="https://www.biologyforlife.com/uploads/2/2/3/9/22392738/c101b0da6ea1a0dab31f80d9963b0368_orig.png" width="60%" height="60%" alt="pokeball">
</div>



In [ ]:
stats = ['hp','attack','defense','sp_attack','sp_defense','speed']

df[stats].hist(figsize=(12,8))

### Check the `is_legendary` column against `base_total` 

The `base_total` column refers to the total power of the Pokémon :

$base\_total = hp + attack + defense + sp\_attack + sp\_defense + speed$

The bar chart compares the **average** `base_total` of legendary and non-legendary Pokémon. It tells us whether being legendary really goes along with being stronger.

In [ ]:
df.groupby("is_legendary")["base_total"].mean().plot(kind='bar')

The histogram shows the overall distribution of `base_total` across all Pokémon, split into 10 bins, with a smooth curve (`kde=True`) on top. It lets us see whether most Pokémon sit in a similar power range and how far the strongest ones are from the rest.

In [ ]:
sns.histplot(df['base_total'],bins=10,kde=True)

# Cleaning

Raw data is rarely ready for modeling. In this section we will remove columns that don't help, fill the missing values, drop duplicates, and limit extreme values, so the data is consistent and safe to use.

### Drop Unnecessary Columns

We need to remove text-heavy, high-cardinality, and irrelevant columns that don't contribute to battle modeling.

- `name` and `japanese_name` are identifiers: every value is unique, so there's no pattern for a model to learn.
- `pokedex_number` is just an ID number.
- `classfication` (spelled this way in the dataset) is a text label with many different values.
- `abilities` is a text list with a huge number of combinations.

Dropping them keeps the dataset small and focused. We don't lose them for good, as the original table is still available in `bk`.

In [ ]:
drop_cols = [
    "japanese_name",
    "name",
    "pokedex_number",
    "classfication",
    "abilities"
]

df = df.drop(columns=drop_cols)


`percentage_male` is dropped in a separate step. It is missing for genderless Pokémon, and gender doesn't affect how strong a Pokémon is in battle, so it isn't worth keeping or filling in.

In [ ]:
df=df.drop(['percentage_male'],axis=1)

### Handle missing values

First we count the missing values in each column (`isna().sum()`), sorted from most to fewest. This tells us which columns need attention. A missing value can't be treated the same way everywhere, so we handle each column according to what the gap actually means.

In [ ]:
df.isna().sum().sort_values(ascending=False)

**Categorical column: `type2`**

Many Pokémon have only one type, so a missing `type2` doesn't mean "unknown data". It simply means "no second type". We fill it with the text `"None"` so that it becomes a real category we can encode later.

In [ ]:
df["type2"] = df["type2"].fillna("None")

**Numerical columns**

Fill numerical data with the median as they're all right skewed. The median is the middle value, so unlike the mean it isn't pulled upward by a few very large values, which makes it a safer "typical value" to fill gaps with.

<div align="center">
    <img src="https://miro.medium.com/v2/resize:fit:1400/1*Abziy_346DuC7teH1Nimtw.png" width="50%" height="50%" alt="pokeball">
</div>



In [ ]:
df["height_m"] = df["height_m"].fillna(df["height_m"].median())

# fill missing values in weight_kg with median ?

`capture_rate` should be a number, but it's stored as text (some entries contain extra words). `pd.to_numeric(..., errors="coerce")` converts what it can and turns anything that can't be converted into a missing value, which we then fill with the median, as above.

In [ ]:
df["capture_rate"] = pd.to_numeric(df["capture_rate"], errors="coerce")
df["capture_rate"] = df["capture_rate"].fillna(df["capture_rate"].median())

### Drop duplicates

We repeat the duplicate check on the cleaned data and remove any repeated rows with `drop_duplicates()`. Doing it after the cleaning steps is worthwhile because dropping columns can make previously different rows identical.

In [ ]:
print("Duplicate check again :",df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicate check after removing again :",df.duplicated().sum())

**Re-check for missing values**

We draw the missing-values heatmap again. If the cleaning worked, no marks should remain: every column should be fully filled.

In [ ]:
sns.heatmap(df.isna())

### Handle Outliers

Outliers are values far away from the rest. We use the **IQR method** on every numeric column:

1. Find the first quartile (`q1`, the 25% mark) and the third quartile (`q3`, the 75% mark).
2. The interquartile range is `iqr = q3 - q1`, which is the spread of the middle half of the data.
3. Anything below `q1 - 1.5 * iqr` or above `q3 + 1.5 * iqr` is treated as an outlier.

<div align="center">
    <img src="https://miro.medium.com/1*0MPDTLn8KoLApoFvI0P2vQ.png" width="60%" height="60%" alt="pokeball">
</div>

Instead of deleting those rows, `clip()` **caps** the values at the lower and upper bounds (Values below lower are replaced with the lower limit. Values above upper are replaced with the upper limit). This way we keep all Pokémon and reduce the pull of extreme values.

The trade-off is that the very strongest (or weakest) Pokémon end up closer to the others than they really are.

In [ ]:
numeric_cols = df.select_dtypes(include=["int64","float64"]).columns

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df[col] = df[col].clip(lower, upper)

# Data Transformation

Machine learning models only understand numbers. In this section we convert the text categories into numeric columns so the model can use them.

### Encoding categorical labels

As they're all non ordinal values (no type is "bigger" than another), simple encoding will work. We use **one-hot encoding**: each category becomes its own column holding 0 or 1. This is better than numbering the categories 1, 2, 3..., which would wrongly tell the model that, say, Water is "greater" than Fire.

First we list the columns that hold categories: `type1`, `type2` and `generation` (it's stored as a number, but it's really a label, as generation 4 isn't "twice" generation 2), plus any True/False columns.

In [ ]:
columns_dummies=["type1","type2","generation"]
columns_dummies.extend(df.select_dtypes(include=["bool"]).columns)

Now we apply the $Dummy\ Encoding$ with `pd.get_dummies` (similar to the One-Hot Encoding). `drop_first=True` removes one column per feature, since it can be deduced from the others (if a Pokémon is none of the listed types, it must be the dropped one), which avoids redundant information. Then we convert `is_legendary` and every True/False column into 0/1 integers, so the whole table is numeric.

In [ ]:
df = pd.get_dummies(df, columns=["type1","type2","generation"], drop_first=True)
df["is_legendary"] = df["is_legendary"].astype(int)

bool_cols = df.select_dtypes(include=["bool"]).columns
# convert boolean columns to int

# Enrich the data

Feature engineering means creating new columns from existing ones to give the model information that's easier to use. Here we build features that describe how a Pokémon behaves in a fight:

- `physical_power` = attack + defense, and `special_power` = sp_attack + sp_defense: how strong it is in physical and special combat.
- `total_combat` = physical + special + speed, which is all its stats except HP.
- `speed_ratio` = speed / base_total: how much of its total power is in speed (fast attackers vs. slow tanks).
- `bulk` = hp + defense + sp_defense: how much damage it can take.
- `avg_type_resistance`: the average of all the `against_*` multipliers (lower means it resists more types).
- `max_weakness`: its highest `against_*` multiplier, meaning the type it's most vulnerable to.

Models don't automatically combine columns on their own, so giving them these summaries can make patterns easier to find.

In [ ]:
df["physical_power"] = df["attack"] + df["defense"]
df["special_power"] = df["sp_attack"] + df["sp_defense"]
df["total_combat"] = df["physical_power"] + df["special_power"] + df["speed"]

df["speed_ratio"] = df["speed"] / df["base_total"]

df["bulk"] = df["hp"] + df["defense"] + df["sp_defense"]

type_cols = [col for col in df.columns if "against_" in col]

df["avg_type_resistance"] = df[type_cols].mean(axis=1)
df["max_weakness"] = df[type_cols].max(axis=1)

# Transforming and scaling numerical values
This is for non Tree based models

Models such as logistic regression, KNN, SVMs and neural networks are sensitive to the scale and skew of the numbers. Tree-based models (like the Random Forest we use later) are not. We do two things:

1. **Log transform** (`np.log1p`, which computes $log(x + 1)$): compresses the long right tail we saw in the histograms. The `+1` makes it safe for values equal to zero.
2. **Standardization** (`StandardScaler`): rescales every column to have a mean of 0 and a standard deviation of 1, so no feature dominates just because its numbers are bigger.

First we separate the **features** `X` (the inputs) from the **target** `y` (what we want to predict, here `is_legendary`).

In [ ]:
# get X and y

A quick look at `X` to confirm it contains all the feature columns and no target column.

In [ ]:
X

Here we apply the two transformations to the numeric columns. We fit the scaler on the data (it learns each column's mean and standard deviation) and transform it in the same step. The result is a NumPy array with no column names.

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = X.select_dtypes(include=["int64","float64"]).columns


for col in numeric_cols: 
    # Apply Log(x + 1) transform to safely handle zeros
    X[col] = np.log1p(X[col])

scaler = StandardScaler()
X = scaler.fit_transform(X[numeric_cols])

Because the scaler returns a plain array, we wrap it back into a DataFrame and re-attach the column names, so it's readable again.

In [ ]:
# Rebuilding the dataset after transformation
df_transformed = pd.DataFrame(X, columns=numeric_cols)

<div style="display: flex; align-items: center; gap: 12px;">
    <h1 style="margin: 0;"><b>Battle Simulation </b></h1>
    <img src="https://cdn-icons-png.flaticon.com/128/7960/7960356.png" width="50" height="50" alt="pokeball">
</div>

The dataset describes individual Pokémon, not battles, so we don't have real examples of who beat whom. To train a model we need to **create labeled battles**: pick two Pokémon, decide a winner with a set of rules, and record the result.

The core idea is the damage formula:

$Damage = (Attack / Defense) * Power * TypeMultiplier$

- **Attack / Defense**: the attacker's offense compared to the defender's defense. A higher ratio means more damage.
- **Power**: how strong the move is. The dataset has no move data, so we use the Pokémon's `speed` as a stand-in.
- **TypeMultiplier**: how effective the attacker's type is against the defender (taken from the `against_*` columns).

#### `simulate_battle`: decide a winner quickly

This function takes two Pokémon (`p1` and `p2`) and returns `1` if the first wins, or `0` if the second wins. Step by step:

1. **Base damage**: apply the formula in both directions.
2. **Look up the types** in the backup table `bk`, because the type columns were encoded away in `df`. `p1.name` is the row's index, which is how we find the same Pokémon in `bk`. We map `"fighting"` to `"fight"` because that's how the column is named (`against_fight`).
3. **Type multiplier**: for each of the attacker's types, read the defender's matching `against_*` value and keep the highest one (a simplification: the attacker uses its best type).
4. **Apply the multiplier** to the damage.
5. **Add randomness** (a random factor between 0.85 and 1.2), so the same matchup doesn't always end the same way, just like real luck in a game.
6. **Final score**: damage plus durability (`hp + sp_defense`). The higher score wins.

In [ ]:
def simulate_battle(p1, p2):

    # Base damage
    p1_damage = (p1["attack"] / p2["defense"]) * p1["speed"]
    p2_damage = (p2["attack"] / p1["defense"]) * p2["speed"]

    # Get types from backup dataset
    p1_types = [str(bk.loc[p1.name, "type1"]).lower(), str(bk.loc[p1.name, "type2"]).lower()]
    p2_types = [str(bk.loc[p2.name, "type1"]).lower(), str(bk.loc[p2.name, "type2"]).lower()]

    type_mapping = {"fighting": "fight"}

    # Compute multipliers (Attacker's types against Defender's resistance)
    p1_multiplier = 1
    for t in p1_types:
        if t != "nan":
            col = f"against_{type_mapping.get(t, t)}"
            p1_multiplier = max(p1_multiplier, p2.get(col, 1))

    p2_multiplier = 1
    for t in p2_types:
        if t != "nan":
            col = f"against_{type_mapping.get(t, t)}"
            p2_multiplier = max(p2_multiplier, p1.get(col, 1))

    # Apply multiplier
    p1_damage *= p1_multiplier
    p2_damage *= p2_multiplier

    # Add randomness
    p1_damage *= np.random.uniform(0.85, 1.2)
    p2_damage *= np.random.uniform(0.85, 1.2)

    # Final score
    p1_score = p1_damage + (p1["hp"] + p1["sp_defense"])
    p2_score = p2_damage + (p2["hp"] + p2["sp_defense"])

    if p1_score > p2_score:
        return 1
    else:
        return 0

#### `generate_battles`: build the training dataset

This function creates the examples the model will learn from. It repeats `n_battles` times:

1. Pick two random Pokémon with `df.sample(1)`.
2. Run `simulate_battle` to find the winner.
3. Build one row containing every feature of both fighters. We add the prefixes `p1_` and `p2_` so the model knows which Pokémon each feature belongs to, and `is_legendary` is left out.
4. Add the `winner` column (our target).

At the end, all the rows are combined into one DataFrame where each row is a single battle.

In [ ]:
def generate_battles(df, n_battles=50000):

    battles = []

    for _ in range(n_battles):

        p1 = df.sample(1).iloc[0]
        p2 = df.sample(1).iloc[0]

        winner = simulate_battle(p1, p2)

        battle_features = {}

        for col in df.columns:
            if col != "is_legendary":
                battle_features[f"p1_{col}"] = p1[col]
                battle_features[f"p2_{col}"] = p2[col]

        battle_features["winner"] = winner

        battles.append(battle_features)

    return pd.DataFrame(battles)

We generate 50,000 battles. More examples give the model more to learn from, but this step loops many times, so it can take a while to run.

**Note:** the battle section uses the unscaled `df`, because the simulator needs the real stat values and the Random Forest doesn't need scaling.

In [ ]:
# Generate battles with df and n_battles=50000

A preview of the battle dataset: each row has the `p1_*` and `p2_*` features plus the `winner` label (1 = Pokémon 1 won, 0 = Pokémon 2 won).

In [ ]:
# Preview the head/sample of the generated battles DataFrame

# Train on the battle dataset

This is **supervised learning**: the model sees many examples where the answer (the winner) is known, and learns the pattern that connects the fighters' features to the outcome. Below we define the features `X` (everything except `winner`) and the target `y` (`winner`).

In [ ]:
X = battle_df.drop("winner", axis=1)
y = battle_df["winner"]

We split the data into a **training set (80%)** and a **test set (20%)** using `train_test_split`. The model learns only from the training set, and we keep the test set hidden so we can later check how it performs on battles it has never seen. `random_state=42` makes the split the same every time we run the notebook.

<div align="center">
    <img src="https://algotrading101.com/learn/wp-content/uploads/2020/06/training-validation-test-data-set.png" width="50%" height="50%" alt="pokeball">
</div>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

We use a **Random Forest**: a collection of many decision trees (`n_estimators=200` means 200 trees), each trained on a random part of the data, that vote on the final answer. It's a strong default for tabular data because it handles non-linear relationships (like attack vs. defense and type effects), and it doesn't need the features to be scaled. `fit` is where the training happens.

<div align="center">
    <img src="https://media.geeksforgeeks.org/wp-content/uploads/20251216121929631349/random_forest_algorithm.webp" width="50%" height="50%" alt="pokeball">
</div>

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Model Evaluation

A model is only useful if it works on data it hasn't seen. We make predictions on the test set and compare them to the real winners.

One thing to keep in mind: the winners were created by our own rule-based simulator, so a high score here shows that the model **learned our simulator's rules**, not that it can predict real Pokémon games.

**Accuracy** is the share of test battles where the model predicted the correct winner.

In [ ]:
preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))

Accuracy alone can hide problems, so we look at more detail:

- The **confusion matrix** counts predictions per outcome. Rows are the real winners and columns are the predicted ones, so the diagonal shows the correct predictions and the other cells show the mistakes.
<div align="center">
    <img src="https://i0.wp.com/glassboxmedicine.com/wp-content/uploads/2019/02/confusion-matrix.png?fit=1200%2C675&ssl=1" width="50%" height="50%" alt="pokeball">
</div>

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['P2 Wins', 'P1 Wins'], yticklabels=['P2 Wins', 'P1 Wins'])
plt.xlabel('Predicted')
plt.ylabel('Actual')

- The **classification report** gives, for each class: **precision** (when the model says "this wins", how often is it right?), **recall** (of all the real wins, how many did it find?), **f1-score** (a balance of the two) and **support** (how many examples there are).

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, preds))

# Battle prediction system

Now we use the trained model on a real matchup chosen by name. The `predict_battle` function:

1. Finds the two Pokémon in the backup table `bk` by name (we removed the names from `df`, and note that the names are case-sensitive, e.g. `"Pikachu"`).
2. Takes their processed rows from `df` using the same row index.
3. Builds one battle row with the same `p1_` / `p2_` columns used in training.
4. Uses `reindex` to make the columns match the training data exactly, in the same names and order, because the model expects that precise layout.
5. Predicts the winner: `1` means the first Pokémon wins, `0` means the second does.

In [ ]:
def predict_battle(pokemon1_name, pokemon2_name):
    
    # Find index using backup dataset
    idx1 = bk[bk["name"] == pokemon1_name].index[0]
    idx2 = bk[bk["name"] == pokemon2_name].index[0]
    
    # Get processed rows from training dataset
    p1 = df.loc[idx1]
    p2 = df.loc[idx2]
    
    #  Build battle feature dictionary
    battle_features = {}
    
    for col in df.columns:
        battle_features[f"p1_{col}"] = p1[col]
        battle_features[f"p2_{col}"] = p2[col]
    
    # Convert to dataframe
    battle_df_test = pd.DataFrame([battle_features])
    
    #  Align columns exactly like training data
    model_columns = battle_df.drop("winner", axis=1).columns
    battle_df_test = battle_df_test.reindex(columns=model_columns, fill_value=0)
    
    # Predict
    prediction = model.predict(battle_df_test)[0]
    
    if prediction == 1:
        print(f"{pokemon1_name} wins!")
    else:
        print(f"{pokemon2_name} wins!")

# Battle simulator

The `battle` function combines both worlds in one place:

1. **AI prediction**: builds the feature row like before, but uses `predict_proba` to get the model's *probability* of each Pokémon winning, which is more informative than just a winner.
2. **Live simulation**: runs `simulate_turn_battle` (defined in the next cell) to play out the fight round by round and prints the battle log.
3. **Result**: prints the winner of the simulated fight.

This lets us compare what the model predicted with what actually happened.

In [ ]:
def battle(pokemon1_name, pokemon2_name):

    # Find index
    idx1 = bk[bk["name"] == pokemon1_name].index[0]
    idx2 = bk[bk["name"] == pokemon2_name].index[0]

    p1 = df.loc[idx1]
    p2 = df.loc[idx2]

    # Build feature row
    battle_features = {}

    for col in df.columns:
        battle_features[f"p1_{col}"] = p1[col]
        battle_features[f"p2_{col}"] = p2[col]

    battle_df_test = pd.DataFrame([battle_features])

    model_columns = battle_df.drop("winner", axis=1).columns
    battle_df_test = battle_df_test.reindex(columns=model_columns, fill_value=0)

    # ML Prediction
    proba = model.predict_proba(battle_df_test)[0]
    p1_win_prob = proba[1]
    p2_win_prob = proba[0]

    print("=== AI Prediction ===")
    print(f"{pokemon1_name} win probability: {p1_win_prob:.2f}")
    print(f"{pokemon2_name} win probability: {p2_win_prob:.2f}")

    # Live Simulation
    sim_result, battle_log = simulate_turn_battle(p1, p2)

    print("\n=== Live Battle Log ===")
    for entry in battle_log:
        print(entry)

    print("\n=== Result ===")
    if sim_result == 1:
        print(f"{pokemon1_name} wins! ⚔️")
    else:
        print(f"{pokemon2_name} wins! ⚔️")

#### `simulate_turn_battle`: play the fight round by round

Unlike `simulate_battle`, which gives a quick verdict, this function plays an actual turn-based fight with HP that goes down until one Pokémon faints:

1. **Copy the HP** into local variables so the original data isn't modified.
2. **Turn order**: the faster Pokémon attacks first (`p1` if there's a tie).
3. **Each turn**, the attacker randomly chooses a **physical** attack (attack vs. defense) or a **special** attack (sp_attack vs. sp_defense), with a base power of 10.
4. **Type multiplier**: the same logic as before, with messages such as "super effective" or "not very effective" added to the log.
5. **Randomness**: a damage factor between 0.85 and 1.0 and a 6% chance of a **critical hit** (x1.5 damage).
6. **Apply the damage** to the defender's HP. If it faints, the battle ends before the other Pokémon can attack.
7. **Repeat** rounds until one HP reaches 0, then return the winner and the full log.

Because this simulation is more detailed than the one used to create the training data (and it has its own randomness), the model's prediction and the live result can disagree, and that's what makes the comparison interesting.

In [ ]:
def simulate_turn_battle(p1, p2):

    # Copy HP so original data not modified
    p1_hp = p1["hp"]
    p2_hp = p2["hp"]

    log = []

    # Map fighting for consistency in dataset naming
    type_mapping = {"fighting": "fight"}

    # Determine turn order
    if p1["speed"] >= p2["speed"]:
        first, second = p1, p2
        first_name, second_name = "P1", "P2"
    else:
        first, second = p2, p1
        first_name, second_name = "P2", "P1"

    round_num = 1

    while p1_hp > 0 and p2_hp > 0:

        log.append(f"\n-- Round {round_num} --")

        # First attacks
        if np.random.rand() < 0.5:
            damage = (first["attack"] / second["defense"]) * 10
            log.append(f"{first_name} uses Physical attack!")
        else:
            damage = (first["sp_attack"] / second["sp_defense"]) * 10
            log.append(f"{first_name} uses Special attack!")

        # Correct type multiplier logic
        first_types = [str(bk.loc[first.name, "type1"]).lower(), str(bk.loc[first.name, "type2"]).lower()]
        second_types = [str(bk.loc[second.name, "type1"]).lower(), str(bk.loc[second.name, "type2"]).lower()]
        
        multiplier = 1
        for t in first_types:
            if t != "nan":
                col = f"against_{type_mapping.get(t, t)}"
                multiplier = max(multiplier, second.get(col, 1))
        
        damage *= multiplier
        if multiplier > 1: log.append("It's super effective! ✨")
        if multiplier < 1 and multiplier != 0: log.append("It's not very effective... 🍃")
        if multiplier == 0: log.append("It had no effect! 😶")

        damage *= np.random.uniform(0.85, 1.0)

        # critical hit
        if np.random.rand() < 0.06:
            damage *= 1.5
            log.append("💥 Critical hit!")

        if first_name == "P1":
            p2_hp -= damage
            log.append(f"P1 deals {damage:.2f} damage.")
        else:
            p1_hp -= damage
            log.append(f"P2 deals {damage:.2f} damage.")

        if p1_hp <= 0 or p2_hp <= 0:
            break

        # Second attacks (Corrected second attacker index)
        if np.random.rand() < 0.5:
            damage = (second["attack"] / first["defense"]) * 10
            log.append(f"{second_name} uses Physical attack!")
        else:
            damage = (second["sp_attack"] / first["sp_defense"]) * 10
            log.append(f"{second_name} uses Special attack!")

        # Correct type multiplier logic
        multiplier = 1
        for t in second_types:
            if t != "nan":
                col = f"against_{type_mapping.get(t, t)}"
                multiplier = max(multiplier, first.get(col, 1))

        damage *= multiplier
        if multiplier > 1: log.append("It's super effective! ✨")
        if multiplier < 1 and multiplier != 0: log.append("It's not very effective... 🍃")
        if multiplier == 0: log.append("It had no effect! 😶")

        damage *= np.random.uniform(0.85, 1.0)

        # critical hit
        if np.random.rand() < 0.06:
            damage *= 1.5
            log.append("💥 Critical hit!")

        if first_name == "P1":
            p1_hp -= damage
            log.append(f"P2 deals {damage:.2f} damage.")
        else:
            p2_hp -= damage
            log.append(f"P1 deals {damage:.2f} damage.")

        round_num += 1

        log.append(f"P1 HP: {max(p1_hp,0):.2f}")
        log.append(f"P2 HP: {max(p2_hp,0):.2f}")

    winner = 1 if p1_hp > 0 else 0

    return winner, log

<div style="display: flex; align-items: center; gap: 12px;">
    <h1 style="margin: 0;"><b>Battle testing</b></h1>
    <img src="https://cdn-icons-png.flaticon.com/128/1752/1752735.png" width="50" height="50" alt="pokeball">
</div>

Time to test everything with three matchups:

- **Pikachu vs. Charizard**: two well-known Pokémon, a good first try.
- **Blastoise vs. Charizard**: a Water type against a Fire type, where the type multiplier should matter.
- **Magikarp vs. Mewtwo**: one of the weakest against one of the strongest, a sanity check in which we'd expect the model to strongly favor Mewtwo.

For each one, compare the model's win probabilities with the winner of the live battle. Since the live battle includes randomness (attack choice, damage rolls, critical hits), you can run a cell again and get a different story, especially in close matchups.

In [ ]:
battle("Pikachu", "Charizard")

In [ ]:
battle("Blastoise", "Charizard")

In [ ]:
battle("Magikarp", "Mewtwo")

<div align="center">

# **Thank you for your attention !**
</div>

